# PEFT multi-epoch manifest

In [1]:
# 24GB card expected (bf16 7B all-linear). This is the real regime, not a tiny smoke.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4090, 24564 MiB


In [7]:
import os
if not os.path.exists('manage.py') and not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

/workspace/Style-Aware-MT/notebooks/Style-Aware-MT
bc694f0


In [3]:
import importlib.util as u
import importlib.metadata as md

pkgs = ["torch", "transformers", "peft", "accelerate",
        "sentence_transformers", "sacrebleu", "yaml", "bitsandbytes"]

pip_name = {"yaml": "PyYAML", "sentence_transformers": "sentence-transformers"}

for m in pkgs:
    if u.find_spec(m):
        try:
            ver = md.version(pip_name.get(m, m))
        except Exception:
            ver = "?"
        print(f"  OK      {m:22} {ver}")
    else:
        print(f"  MISSING {m}")

  OK      torch                  2.12.0
  OK      transformers           5.12.1
  OK      peft                   0.18.0
  OK      accelerate             1.14.0
  OK      sentence_transformers  5.5.1
  OK      sacrebleu              2.6.0
  OK      yaml                   6.0.3
  OK      bitsandbytes           0.49.2


In [13]:
import getpass, os

os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")
print("HF_TOKEN set for this kernel:", "yes" if os.environ.get("HF_TOKEN") else "no")

HF_TOKEN:  ········


HF_TOKEN set for this kernel: yes


## Step A — build the single anchor-cell config 

In [19]:
import yaml
from src.peft.sweep import build_cell_config

base = yaml.safe_load(open("configs/peft_sweep.yaml"))
anchor = next(c for c in base["sweep"]["grid"] if c.get("anchor"))
cfg, out_dir = build_cell_config(base, anchor, base["sweep"]["output_base"])

print("anchor cell:", anchor, "-> output_dir:", out_dir)
t = cfg["peft"]["train"]
print(f"epochs={t['num_train_epochs']}  data.limit={cfg['data']['limit']}  "
      f"load_best={t.get('load_best_model_at_end')}  save_total_limit={t.get('save_total_limit')}")
assert t["num_train_epochs"] == 3,          "expected the real 3-epoch count"
assert cfg["data"]["limit"] is None,        "expected full data (no smoke limit)"
assert t.get("load_best_model_at_end") is False, "manifest is only written when load_best is false"

with open("configs/peft_anchor_e3.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("wrote configs/peft_anchor_e3.yaml")

anchor cell: {'r': 16, 'alpha': 32, 'lr': 0.0002, 'anchor': True} -> output_dir: models/peft_lora_r16_lr2e-4
epochs=3  data.limit=None  load_best=False  save_total_limit=None
wrote configs/peft_anchor_e3.yaml


## Step B 


In [21]:
!python3 -m src.peft.train --config configs/peft_anchor_e3.yaml

Loading weights: 100%|█████████████████████| 339/339 [00:00<00:00, 4158.86it/s]
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
Tokenizing 10860 train / 1323 dev examples ...
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '2.789', 'grad_norm': '3.278', 'learning_rate': '2.903e-05', 'epoch': '0.01473'}
{'loss': '2.059', 'grad_norm': '1.596', 'learning_rate': '6.129e-05', 'epoch': '0.02947'}
{'loss': '1.73', 'grad_norm': '1.444', 'learning_rate': '9.355e-05', 'epoch': '0.0442'}
{'loss': '1.68', 'grad_norm': '1.429', 'learning_rate': '0.0001258', 'epoch': '0.05893'}
{'loss': '1.664', 'grad_norm': '1.576', 'learning_rate': '0.0001581', 'epoch': '0.07366'}
{'loss': '1.549', 'grad_norm': '1.323', 'learning_rate': '0.0001903', 'epoch': '0.0884'}
{'loss': '1.522', 'grad_norm': '1.262', 'learning_rate': '0.0002', 'epoch': '0.1031'}
{'loss': '1.518', 'grad_norm': '1.195', 'learning_rate': '0.0002', 'epoc

## Step C — the manifest check 

In [22]:
import json
from pathlib import Path

man_path = Path("models/peft_lora_r16_lr2e-4/epoch_checkpoints.json")
print(man_path.read_text())          # the `cat`

man = json.loads(man_path.read_text())
epochs = [m["epoch"] for m in man]
steps  = [m["step"] for m in man]
print("epochs:", epochs, " steps:", steps)

assert len(man) == 3,          f"expected 3 epoch entries, got {len(man)} -- multi-epoch manifest broken"
assert epochs == [1, 2, 3],    f"epochs not distinct/ordered 1..3: {epochs}"
assert len(set(steps)) == 3,   f"duplicate checkpoint step -- the e3-duplicate bug is back: {steps}"
assert all(Path(m["checkpoint"]).exists() for m in man), "a manifest checkpoint dir is missing on disk"
print("\nMANIFEST OK: 3 distinct epoch checkpoints, no duplicate e3 -- snapshot+dedup holds on a real run.")

[
  {
    "epoch": 1,
    "step": 679,
    "eval_loss": 1.5333021879196167,
    "checkpoint": "models/peft_lora_r16_lr2e-4/checkpoint-679"
  },
  {
    "epoch": 2,
    "step": 1358,
    "eval_loss": 1.5591294765472412,
    "checkpoint": "models/peft_lora_r16_lr2e-4/checkpoint-1358"
  },
  {
    "epoch": 3,
    "step": 2037,
    "eval_loss": 1.7798848152160645,
    "checkpoint": "models/peft_lora_r16_lr2e-4/checkpoint-2037"
  }
]
epochs: [1, 2, 3]  steps: [679, 1358, 2037]

MANIFEST OK: 3 distinct epoch checkpoints, no duplicate e3 -- snapshot+dedup holds on a real run.


## Step D — eval_loss trajectory 

In [23]:
losses = [(m["epoch"], m["eval_loss"]) for m in man]
print("eval_loss by epoch:")
for ep, el in losses:
    print(f"  epoch {ep}: {el:.4f}")

vals = [el for _, el in losses]
assert len(set(round(v, 6) for v in vals)) > 1, \
    "eval_loss identical across epochs -- training is not updating the model"

print(f"\ndelta  e1->e2: {vals[1]-vals[0]:+.4f}   e2->e3: {vals[2]-vals[1]:+.4f}")
if vals[1] < vals[0]:
    tail = "e2->e3 rose = overfitting onset (expected, fine)." if vals[2] > vals[1] \
           else "e2->e3 still dropping."
    print("e1->e2 dropped (learning). " + tail)
else:
    print("WARNING: eval_loss did not drop e1->e2 -- inspect LR / masking / grads before the full sweep.")

eval_loss by epoch:
  epoch 1: 1.5333
  epoch 2: 1.5591
  epoch 3: 1.7799

delta  e1->e2: +0.0258   e2->e3: +0.2208


In [ ]:
git add .
    